# Train prompt LoRA on Colab T4
Set **Runtime → Change runtime type → T4 GPU**, then run all cells.

In [ ]:
!pip -q install transformers peft trl bitsandbytes datasets accelerate httpx

In [ ]:
from pathlib import Path
from google.colab import files

print("Upload train_qlora.py, eval_prompts.py, distill.py, data/train_structured.jsonl, and data/val_structured.jsonl")
uploaded = files.upload()
Path("data").mkdir(exist_ok=True)
Path("eval/out").mkdir(parents=True, exist_ok=True)
Path("train_qlora.py").write_bytes(uploaded["train_qlora.py"])
Path("eval_prompts.py").write_bytes(uploaded["eval_prompts.py"])
Path("distill.py").write_bytes(uploaded["distill.py"])
Path("data/train_structured.jsonl").write_bytes(uploaded["train_structured.jsonl"])
Path("data/val_structured.jsonl").write_bytes(uploaded["val_structured.jsonl"])

In [ ]:
!python eval_prompts.py --skip-judge --name base

In [ ]:
!python train_qlora.py

In [ ]:
!python eval_prompts.py --skip-judge --adapter out/adapter --name adapter

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

staging_dir = Path("prompt-lora-run")
if staging_dir.exists():
    shutil.rmtree(staging_dir)
shutil.copytree("out/adapter", staging_dir / "out/adapter")
(staging_dir / "eval/out").mkdir(parents=True)
for output in Path("eval/out").glob("*.json"):
    shutil.copy2(output, staging_dir / "eval/out" / output.name)
shutil.make_archive("prompt-lora-run", "zip", staging_dir)
files.download("prompt-lora-run.zip")